# Búsqueda en espacios de estados

Muchos puzles se resuelven igual: cada situación es un **estado**, desde cada estado se llega a otros con un movimiento (sus **sucesores**) y algunos estados son **finales**. Con esa descripción, unas pocas funciones genéricas buscan la solución de cualquiera de ellos: en anchura (la de menos movimientos), en profundidad, por escalada y primero el mejor, estas dos guiadas por una **heurística** que estima lo lejos que está la solución. Es BFS o DFS sobre un grafo que no se construye de antemano, sino que se genera según hace falta.

Con factor de ramificación $b$ y la solución a profundidad $d$, buscar sin recordar los estados visitados genera $O(b^d)$ estados; recordándolos, cada estado se expande una vez como mucho y el coste es $O(|E| + |T|)$, con $|E|$ estados alcanzables y $|T|$ movimientos entre ellos. Al final, un esquema genérico de divide y vencerás y el rompecabezas del triominó. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y [Edabit](https://edabit.com/challenges/python3).

## 1. El esquema de un problema

Definir una `dataclass` `Problema` con el estado inicial, la función `sucesores(estado)`, el predicado `es_final(estado)` y, opcionalmente, `heuristica(estado)`, y otra `Resultado` con el camino encontrado (la lista de estados desde el inicial, o `None`) y el número de estados expandidos. Escribir `es_camino_valido(problema, camino)` y describir con este esquema el problema del contador: empieza en 1, solo tiene los botones `+1` y `×2` y hay que llegar exactamente a $n$ sin pasarse.

In [1]:
import heapq
import itertools
import math
import random
import string
from collections import Counter, defaultdict, deque
from collections.abc import Callable, Hashable, Iterable
from dataclasses import dataclass


@dataclass
class Problema:
    inicial: Hashable
    sucesores: Callable[[Hashable], Iterable[Hashable]]
    es_final: Callable[[Hashable], bool]
    heuristica: Callable[[Hashable], float] | None = None


@dataclass
class Resultado:
    camino: list | None
    expandidos: int
    atascado: bool = False

    @property
    def movimientos(self):
        return None if self.camino is None else len(self.camino) - 1


def es_camino_valido(problema, camino):
    return (camino[0] == problema.inicial and problema.es_final(camino[-1])
            and all(b in problema.sucesores(a) for a, b in itertools.pairwise(camino)))


def contador(n):
    return Problema(
        inicial=1,
        sucesores=lambda x: [y for y in sorted({x + 1, 2 * x}) if y <= n],
        es_final=lambda x: x == n,
        heuristica=lambda x: n - x,
    )


p = contador(10)
for x in (1, 3, 6, 9):
    print(f"sucesores({x}) = {p.sucesores(x)}")
assert es_camino_valido(p, [1, 2, 4, 5, 10])
assert not es_camino_valido(p, [1, 2, 4, 8, 10])
assert not es_camino_valido(p, [1, 2, 4, 5])

sucesores(1) = [2]
sucesores(3) = [4, 6]
sucesores(6) = [7]
sucesores(9) = [10]


## 2. Búsqueda en anchura

Escribir `busqueda_anchura(problema)`: los estados por expandir esperan en una cola y un diccionario guarda el padre de cada estado ya visto, que sirve a la vez para no repetir estados y para reconstruir el camino. Devuelve el camino con menos movimientos. Comprobar con el contador que el mínimo de pulsaciones para llegar a $n$ es $(\text{cifras binarias de } n - 1) + (\text{unos de } n - 1)$: cada `×2` añade una cifra binaria y cada `+1` pone a uno la última.

In [2]:
def reconstruye(padres, estado):
    camino = []
    while estado is not None:
        camino.append(estado)
        estado = padres[estado]
    return camino[::-1]


def busqueda_anchura(problema):
    padres = {problema.inicial: None}
    cola = deque([problema.inicial])
    expandidos = 0
    while cola:
        estado = cola.popleft()
        if problema.es_final(estado):
            return Resultado(reconstruye(padres, estado), expandidos)
        expandidos += 1
        for siguiente in problema.sucesores(estado):
            if siguiente not in padres:
                padres[siguiente] = estado
                cola.append(siguiente)
    return Resultado(None, expandidos)


r = busqueda_anchura(contador(2026))
print(f"2026 = {2026:b} en binario")
print(f"Camino: {r.camino}")
print(f"{r.movimientos} pulsaciones, {r.expandidos} estados expandidos")

for n in range(1, 600):
    r = busqueda_anchura(contador(n))
    assert es_camino_valido(contador(n), r.camino)
    assert r.movimientos == (n.bit_length() - 1) + (n.bit_count() - 1)

2026 = 11111101010 en binario
Camino: [1, 2, 3, 6, 7, 14, 15, 30, 31, 62, 63, 126, 252, 253, 506, 1012, 1013, 2026]
17 pulsaciones, 1864 estados expandidos


## 3. Búsqueda en profundidad con control de repetidos

Escribir `busqueda_profundidad(problema)` con una pila en lugar de la cola y el mismo diccionario de padres: sin él, en cuanto un movimiento se pueda deshacer, la búsqueda daría vueltas para siempre. Los sucesores se apilan al revés para explorar primero el primero. Comprobar con el contador que el camino es válido, pero no el más corto, y que depende del orden en que se generan los sucesores.

In [3]:
def busqueda_profundidad(problema):
    padres = {problema.inicial: None}
    pila = [problema.inicial]
    expandidos = 0
    while pila:
        estado = pila.pop()
        if problema.es_final(estado):
            return Resultado(reconstruye(padres, estado), expandidos)
        expandidos += 1
        for siguiente in reversed(list(problema.sucesores(estado))):
            if siguiente not in padres:
                padres[siguiente] = estado
                pila.append(siguiente)
    return Resultado(None, expandidos)


p = contador(2026)
doble_primero = Problema(1, lambda x: p.sucesores(x)[::-1], p.es_final)
print(f"{'Búsqueda':<30}{'movimientos':>12}{'expandidos':>12}")
for nombre, problema, buscar in [("anchura", p, busqueda_anchura),
                                 ("profundidad, +1 primero", p, busqueda_profundidad),
                                 ("profundidad, ×2 primero", doble_primero, busqueda_profundidad)]:
    r = buscar(problema)
    assert es_camino_valido(problema, r.camino)
    print(f"{nombre:<30}{r.movimientos:>12}{r.expandidos:>12}")

Búsqueda                       movimientos  expandidos
anchura                                 17        1864
profundidad, +1 primero               1119        1119
profundidad, ×2 primero               1012        1012


## 4. Escalada

Escribir `escalada(problema)`: desde el estado actual pasa al sucesor de menor heurística, pero solo si mejora la del actual; si ninguno mejora, se para y marca el resultado como atascado. No recuerda los estados visitados, así que apenas gasta memoria, pero se queda en el primer óptimo local. Probarla en el perfil de alturas de un sendero que cruza una isla: el estado es la posición, los sucesores son las posiciones vecinas y la heurística es lo que falta para la cumbre. Ver desde qué salidas se llega a la cumbre y desde cuáles se queda en un pico secundario.

In [4]:
def escalada(problema):
    h = problema.heuristica
    estado, camino, expandidos = problema.inicial, [problema.inicial], 0
    while not problema.es_final(estado):
        expandidos += 1
        mejor = min(problema.sucesores(estado), key=h, default=None)
        if mejor is None or h(mejor) >= h(estado):
            return Resultado(camino, expandidos, atascado=True)
        estado = mejor
        camino.append(estado)
    return Resultado(camino, expandidos)


alturas = [0, 150, 420, 640, 580, 470, 690, 1250, 2050, 2900, 3715, 3100, 2200, 1400, 1150, 1320, 1290, 700, 250, 0]
cumbre = max(alturas)


def sendero(salida):
    return Problema(
        inicial=salida,
        sucesores=lambda i: [j for j in (i - 1, i + 1) if 0 <= j < len(alturas)],
        es_final=lambda i: alturas[i] == cumbre,
        heuristica=lambda i: cumbre - alturas[i],
    )


llegadas = defaultdict(list)
for salida in range(len(alturas)):
    r = escalada(sendero(salida))
    fin = r.camino[-1]
    llegadas[fin].append(salida)
    assert all(alturas[v] <= alturas[fin] for v in sendero(fin).sucesores(fin))
    assert r.atascado == (alturas[fin] != cumbre)

for fin, salidas in sorted(llegadas.items()):
    estado = "cumbre" if alturas[fin] == cumbre else "atascada en un pico secundario"
    print(f"Posición {fin:>2} ({alturas[fin]:>4} m, {estado}): desde las salidas {salidas}")

Posición  3 ( 640 m, atascada en un pico secundario): desde las salidas [0, 1, 2, 3, 4]
Posición 10 (3715 m, cumbre): desde las salidas [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Posición 15 (1320 m, atascada en un pico secundario): desde las salidas [15, 16, 17, 18, 19]


## 5. Primero el mejor

Escribir `primero_el_mejor(problema)`: como la búsqueda en anchura, pero los estados por expandir esperan en un montículo (`heapq`) ordenado por la heurística y siempre se expande el que parece más cercano a la solución; un contador desempata para no tener que comparar estados. Compararla con la búsqueda en anchura en el contador con dos heurísticas: $n - x$, que empuja a doblar cuanto antes, y el número exacto de pulsaciones que faltan, que se obtiene yendo hacia atrás desde $n$ (dividir entre 2 mientras se pueda sin bajar de $x$ y, si no, restar 1).

In [5]:
def primero_el_mejor(problema):
    h, desempate = problema.heuristica, itertools.count()
    padres = {problema.inicial: None}
    abiertos = [(h(problema.inicial), next(desempate), problema.inicial)]
    expandidos = 0
    while abiertos:
        _, _, estado = heapq.heappop(abiertos)
        if problema.es_final(estado):
            return Resultado(reconstruye(padres, estado), expandidos)
        expandidos += 1
        for siguiente in problema.sucesores(estado):
            if siguiente not in padres:
                padres[siguiente] = estado
                heapq.heappush(abiertos, (h(siguiente), next(desempate), siguiente))
    return Resultado(None, expandidos)


def pulsaciones_que_faltan(x, n):
    pulsaciones = 0
    while n > x:
        n = n // 2 if n % 2 == 0 and n // 2 >= x else n - 1
        pulsaciones += 1
    return pulsaciones


n = 2026
p = contador(n)
exacta = Problema(1, p.sucesores, p.es_final, lambda x: pulsaciones_que_faltan(x, n))
print(f"{'Búsqueda':<32}{'movimientos':>12}{'expandidos':>12}")
for nombre, problema, buscar in [("anchura", p, busqueda_anchura),
                                 ("primero el mejor, n - x", p, primero_el_mejor),
                                 ("primero el mejor, exacta", exacta, primero_el_mejor)]:
    r = buscar(problema)
    assert es_camino_valido(problema, r.camino)
    print(f"{nombre:<32}{r.movimientos:>12}{r.expandidos:>12}")

assert pulsaciones_que_faltan(1, n) == busqueda_anchura(p).movimientos
assert primero_el_mejor(exacta).expandidos == primero_el_mejor(exacta).movimientos

Búsqueda                         movimientos  expandidos
anchura                                   17        1864
primero el mejor, n - x                 1012        1012
primero el mejor, exacta                  17          17


## 6. El granjero, el lobo, la cabra y la col

Un granjero tiene que cruzar un barranco por una pasarela con un lobo, una cabra y una col, y solo puede llevar consigo a uno de ellos cada vez. Si los deja solos, el lobo se come la cabra y la cabra se come la col. Representar cada estado por el conjunto de los que están en la orilla izquierda, resolverlo con la búsqueda en anchura y contar los planes con el mínimo de cruces.

In [6]:
TODOS = frozenset({"granjero", "lobo", "cabra", "col"})


def es_seguro(izquierda):
    for orilla in (izquierda, TODOS - izquierda):
        if "granjero" not in orilla and ({"lobo", "cabra"} <= orilla or {"cabra", "col"} <= orilla):
            return False
    return True


def sucesores_granjero(izquierda):
    en_izquierda = "granjero" in izquierda
    orilla = izquierda if en_izquierda else TODOS - izquierda
    for acompañante in [None, *sorted(orilla - {"granjero"})]:
        cruzan = {"granjero", acompañante} - {None}
        nuevo = izquierda - cruzan if en_izquierda else izquierda | cruzan
        if es_seguro(nuevo):
            yield nuevo


granjero = Problema(TODOS, sucesores_granjero, lambda izquierda: not izquierda)


CON = {"lobo": "con el lobo", "cabra": "con la cabra", "col": "con la col"}


def describe(antes, despues):
    acompañantes = [CON[a] for a in (antes ^ despues) - {"granjero"}]
    sentido = "→" if "granjero" in antes else "←"
    return f"{sentido} el granjero cruza {acompañantes[0] if acompañantes else 'solo'}"


def planes_minimos(problema, movimientos):
    def extiende(camino):
        if len(camino) == movimientos + 1:
            if problema.es_final(camino[-1]):
                yield camino
            return
        for siguiente in problema.sucesores(camino[-1]):
            if siguiente not in camino:
                yield from extiende(camino + [siguiente])
    return list(extiende([problema.inicial]))


r = busqueda_anchura(granjero)
assert es_camino_valido(granjero, r.camino)
for antes, despues in itertools.pairwise(r.camino):
    print(describe(antes, despues))

planes = planes_minimos(granjero, r.movimientos)
print(f"\n{r.movimientos} cruces; planes distintos con ese mínimo: {len(planes)}")
assert r.movimientos == 7 and len(planes) == 2
assert all(es_seguro(e) for plan in planes for e in plan)

→ el granjero cruza con la cabra
← el granjero cruza solo
→ el granjero cruza con la col
← el granjero cruza con la cabra
→ el granjero cruza con el lobo
← el granjero cruza solo
→ el granjero cruza con la cabra

7 cruces; planes distintos con ese mínimo: 2


## 7. Jarras generalizadas

Generalizar el problema de las jarras (que en *Puzles* se resuelve para 3 y 5 litros) a cualquier número de jarras sin marcas con capacidades dadas. Con grifo, cada movimiento llena una jarra, la vacía o vierte una en otra hasta que la primera se vacía o la segunda se llena; sin grifo, solo se vierte. Resolver: medir 1 litro con jarras de 6, 10 y 15; repartir a partes iguales 8 litros de vino entre jarras de 8, 5 y 3 sin grifo. Comprobar con capacidades al azar que con dos o tres jarras y grifo se puede medir $x$ litros si y solo si $x$ es múltiplo del máximo común divisor de las capacidades y no pasa de la mayor.

In [7]:
def problema_jarras(capacidades, es_final, inicial=None, grifo=True):
    n = len(capacidades)

    def sucesores(estado):
        for i in range(n):
            if grifo:
                yield estado[:i] + (capacidades[i],) + estado[i + 1:]
                yield estado[:i] + (0,) + estado[i + 1:]
            for j in range(n):
                cantidad = min(estado[i], capacidades[j] - estado[j]) if i != j else 0
                if cantidad:
                    nuevo = list(estado)
                    nuevo[i] -= cantidad
                    nuevo[j] += cantidad
                    yield tuple(nuevo)

    return Problema(inicial or (0,) * n, sucesores, es_final)


def muestra(r):
    print(f"{r.movimientos} movimientos: " + " → ".join(str(e) for e in r.camino))


medir_1 = problema_jarras((6, 10, 15), lambda e: 1 in e)
muestra(busqueda_anchura(medir_1))

vino = problema_jarras((8, 5, 3), lambda e: e == (4, 4, 0), inicial=(8, 0, 0), grifo=False)
r = busqueda_anchura(vino)
muestra(r)
assert r.movimientos == 7 and es_camino_valido(vino, r.camino)
assert all(sum(e) == 8 for e in r.camino)


def medibles(capacidades):
    medibles = set()
    for objetivo in range(max(capacidades) + 1):
        if busqueda_anchura(problema_jarras(capacidades, lambda e: objetivo in e)).camino:
            medibles.add(objetivo)
    return medibles


random.seed(7)
for _ in range(60):
    capacidades = tuple(random.randint(1, 12) for _ in range(random.choice((2, 3))))
    g = math.gcd(*capacidades)
    assert medibles(capacidades) == set(range(0, max(capacidades) + 1, g)), capacidades
print("Condición del máximo común divisor comprobada en 60 casos al azar")

4 movimientos: (0, 0, 0) → (6, 0, 0) → (0, 0, 6) → (0, 10, 6) → (0, 1, 15)
7 movimientos: (8, 0, 0) → (3, 5, 0) → (3, 2, 3) → (6, 2, 0) → (6, 0, 2) → (1, 5, 2) → (1, 4, 3) → (4, 4, 0)
Condición del máximo común divisor comprobada en 60 casos al azar


## 8. El 8-puzle con dos heurísticas

El 8-puzle tiene ocho fichas numeradas y un hueco en un tablero de $3 \times 3$; cada movimiento desliza al hueco una ficha vecina. Resolverlo con la búsqueda en anchura (el mínimo de movimientos) y con primero el mejor usando dos heurísticas: el número de fichas mal colocadas y la suma de las distancias Manhattan de cada ficha a su sitio. Comparar movimientos y estados expandidos en varios tableros desordenados con paseos al azar (sin deshacer el último movimiento), y ver si la escalada con la distancia Manhattan llega a la solución o se atasca. Comprobar que la distancia Manhattan nunca es menor que las fichas mal colocadas.

In [8]:
OBJETIVO = (1, 2, 3, 4, 5, 6, 7, 8, 0)
VECINAS = {i: [j for j in range(9) if abs(i // 3 - j // 3) + abs(i % 3 - j % 3) == 1] for i in range(9)}


def sucesores_puzle(estado):
    hueco = estado.index(0)
    for j in VECINAS[hueco]:
        nuevo = list(estado)
        nuevo[hueco], nuevo[j] = nuevo[j], 0
        yield tuple(nuevo)


def mal_colocadas(estado):
    return sum(1 for ficha, sitio in zip(estado, OBJETIVO) if ficha and ficha != sitio)


def manhattan(estado):
    return sum(abs(i // 3 - (f - 1) // 3) + abs(i % 3 - (f - 1) % 3) for i, f in enumerate(estado) if f)


def ocho_puzle(inicial, heuristica=None):
    return Problema(inicial, sucesores_puzle, lambda e: e == OBJETIVO, heuristica)


def dibuja(estado):
    return "\n".join(" ".join(str(f) if f else "·" for f in estado[i:i + 3]) for i in (0, 3, 6))


def desordena(pasos, rng):
    anterior, estado = None, OBJETIVO
    for _ in range(pasos):
        anterior, estado = estado, rng.choice([s for s in sucesores_puzle(estado) if s != anterior])
    return estado


rng = random.Random(2026)
tableros = [desordena(pasos, rng) for pasos in (6, 12, 18, 24, 50, 200)]
print(dibuja(tableros[2]), end="\n\n")

print(f"{'BFS':>9}{'mal colocadas':>18}{'Manhattan':>18}   escalada")
for tablero in tableros:
    anchura = busqueda_anchura(ocho_puzle(tablero))
    fila = f"{anchura.movimientos:>3} {anchura.expandidos:>6}"
    for h in (mal_colocadas, manhattan):
        r = primero_el_mejor(ocho_puzle(tablero, h))
        assert es_camino_valido(ocho_puzle(tablero), r.camino) and r.movimientos >= anchura.movimientos
        fila += f"{r.movimientos:>9} {r.expandidos:>8}"
    e = escalada(ocho_puzle(tablero, manhattan))
    fila += f"   {'atascada' if e.atascado else 'llega'} tras {e.movimientos} movimientos"
    print(fila)
print("(cada columna: movimientos y estados expandidos)")

for _ in range(2000):
    estado = tuple(rng.sample(range(9), 9))
    assert manhattan(estado) >= mal_colocadas(estado)

1 3 4
2 5 6
7 8 ·

      BFS     mal colocadas         Manhattan   escalada
  6     97        6       39        6        6   llega tras 6 movimientos
 12   1449       48      698       42      362   atascada tras 4 movimientos
 18  18281       22      196       18      235   atascada tras 0 movimientos
 16  11350       54      976       38      100   atascada tras 4 movimientos


 22  93758       64      904       62      651   atascada tras 2 movimientos


 24 130400       48      621       52      119   atascada tras 5 movimientos
(cada columna: movimientos y estados expandidos)


## 9. Tableros sin solución

Una inversión es una pareja de fichas que aparecen en orden contrario al del objetivo (sin contar el hueco). Un movimiento horizontal no cambia el orden de las fichas y uno vertical pasa una ficha por delante o por detrás de otras dos, así que la paridad del número de inversiones nunca cambia: los tableros con un número impar de inversiones no tienen solución. Escribir `tiene_solucion(estado)` y comprobarlo con todas las $9!$ colocaciones, calculando los estados alcanzables desde el objetivo. Ver cuánto trabajo cuesta a la búsqueda en anchura descubrir que no hay solución si se intercambian el 7 y el 8.

In [9]:
def inversiones(estado):
    fichas = [f for f in estado if f]
    return sum(a > b for a, b in itertools.combinations(fichas, 2))


def tiene_solucion(estado):
    return inversiones(estado) % 2 == 0


def alcanzables(inicial, sucesores):
    vistos, cola = {inicial}, deque([inicial])
    while cola:
        for siguiente in sucesores(cola.popleft()):
            if siguiente not in vistos:
                vistos.add(siguiente)
                cola.append(siguiente)
    return vistos


resolubles = alcanzables(OBJETIVO, sucesores_puzle)
print(f"Alcanzables desde el objetivo: {len(resolubles)} = 9!/2 = {math.factorial(9) // 2}")
assert all((estado in resolubles) == tiene_solucion(estado) for estado in itertools.permutations(range(9)))

cambiado = (1, 2, 3, 4, 5, 6, 8, 7, 0)
print(dibuja(cambiado))
r = busqueda_anchura(ocho_puzle(cambiado))
print(f"Inversiones: {inversiones(cambiado)}; tiene solución: {tiene_solucion(cambiado)}")
print(f"La búsqueda en anchura expande {r.expandidos} estados para devolver {r.camino}")
assert r.camino is None and r.expandidos == len(resolubles)

Alcanzables desde el objetivo: 181440 = 9!/2 = 181440


1 2 3
4 5 6
8 7 ·
Inversiones: 1; tiene solución: False
La búsqueda en anchura expande 181440 estados para devolver None


## 10. Fichas blancas y negras

En una fila hay $n$ fichas blancas a la izquierda, un hueco y $n$ fichas negras a la derecha (`BBB_NNN`). Las blancas solo avanzan hacia la derecha y las negras hacia la izquierda, y cada movimiento desliza una ficha al hueco contiguo o la hace saltar al hueco por encima de una ficha del otro color. Hay que intercambiar los dos grupos (`NNN_BBB`). Resolverlo con la búsqueda en anchura y en profundidad, comparar los estados expandidos y comprobar que siempre hacen falta $n^2 + 2n$ movimientos: cada blanca salta una vez sobre cada negra ($n^2$ saltos) y, como cada ficha avanza $n + 1$ casillas, quedan $2n$ deslizamientos.

In [10]:
def sucesores_fichas(estado):
    hueco = estado.index("_")
    for origen, ficha in ((hueco - 1, "B"), (hueco - 2, "B"), (hueco + 1, "N"), (hueco + 2, "N")):
        if 0 <= origen < len(estado) and estado[origen] == ficha:
            if abs(origen - hueco) == 2 and estado[(origen + hueco) // 2] == ficha:
                continue
            nuevo = list(estado)
            nuevo[hueco], nuevo[origen] = ficha, "_"
            yield "".join(nuevo)


def fichas(n):
    return Problema("B" * n + "_" + "N" * n, sucesores_fichas, lambda e: e == "N" * n + "_" + "B" * n)


print(" → ".join(busqueda_anchura(fichas(3)).camino))
print(f"\n{'n':>2}{'movimientos':>13}{'anchura':>10}{'profundidad':>13}")
for n in range(1, 8):
    anchura, profundidad = busqueda_anchura(fichas(n)), busqueda_profundidad(fichas(n))
    assert anchura.movimientos == profundidad.movimientos == n * n + 2 * n
    assert es_camino_valido(fichas(n), profundidad.camino)
    print(f"{n:>2}{anchura.movimientos:>13}{anchura.expandidos:>10}{profundidad.expandidos:>13}")

BBB_NNN → BB_BNNN → BBNB_NN → BBNBN_N → BBN_NBN → B_NBNBN → _BNBNBN → NB_BNBN → NBNB_BN → NBNBNB_ → NBNBN_B → NBN_NBB → N_NBNBB → NN_BNBB → NNNB_BB → NNN_BBB

 n  movimientos   anchura  profundidad
 1            3         5            3
 2            8        22           14
 3           15        71           34
 4           24       194           82
 5           35       475          165
 6           48      1088          352
 7           63      2387          690


## 11. Cadena de dominó

Colocar todas las fichas de dominó de una lista en una sola línea de modo que los números de las mitades que se tocan coincidan (las fichas se pueden girar). El estado es la cadena construida y las fichas que faltan; los sucesores añaden al final una ficha que encaje. Resolverlo con la búsqueda en profundidad y comprobarlo con la teoría de grafos: si los números son vértices y las fichas aristas, hay cadena si y solo si las fichas forman un grafo conexo con 0 o 2 vértices de grado impar (un camino euleriano).

In [11]:
def problema_domino(fichas):
    def sucesores(estado):
        cadena, faltan = estado
        for a, b in sorted(faltan):
            for girada in [(a, b)] if a == b else [(a, b), (b, a)]:
                if not cadena or cadena[-1][1] == girada[0]:
                    yield cadena + (girada,), faltan - {(a, b)}

    return Problema(((), frozenset(tuple(sorted(f)) for f in fichas)), sucesores, lambda e: not e[1])


def cadena_domino(fichas):
    r = busqueda_profundidad(problema_domino(fichas))
    return r.camino[-1][0] if r.camino else None


def hay_camino_euleriano(fichas):
    grado, vecinos = Counter(), defaultdict(set)
    for a, b in fichas:
        grado[a] += 1
        grado[b] += 1
        vecinos[a].add(b)
        vecinos[b].add(a)
    inicio = next(iter(grado))
    vistos = alcanzables(inicio, lambda v: vecinos[v])
    return len(vistos) == len(grado) and sum(g % 2 for g in grado.values()) in (0, 2)


mano = [(3, 5), (0, 1), (6, 6), (2, 6), (1, 3), (5, 2), (3, 3), (4, 6)]
cadena = cadena_domino(mano)
print("".join(f"[{a}|{b}]" for a, b in cadena))
assert sorted(tuple(sorted(f)) for f in cadena) == sorted(tuple(sorted(f)) for f in mano)
assert all(x[1] == y[0] for x, y in itertools.pairwise(cadena))
print("Con [1|2] y [3|4]:", cadena_domino([(1, 2), (3, 4)]))

juego = [(a, b) for a in range(7) for b in range(a, 7)]
random.seed(11)
con_cadena = 0
for _ in range(300):
    fichas_al_azar = random.sample(juego, 6)
    cadena = cadena_domino(fichas_al_azar)
    assert (cadena is not None) == hay_camino_euleriano(fichas_al_azar)
    con_cadena += cadena is not None
print(f"De 300 manos de 6 fichas al azar, {con_cadena} se pueden encadenar")

[0|1][1|3][3|3][3|5][5|2][2|6][6|6][6|4]
Con [1|2] y [3|4]: None
De 300 manos de 6 fichas al azar, 57 se pueden encadenar


## 12. Calendario de una liga

Organizar una liga a una vuelta de $n$ equipos en $n - 1$ jornadas con el método del círculo: un equipo queda fijo y los demás giran una posición en cada jornada, y en cada una se enfrentan el primero con el último, el segundo con el penúltimo, etc. Si $n$ es impar, se añade un equipo ficticio, que hace de equipo fijo, y quien juega contra él descansa. Alternar quién juega en casa para que cada equipo tenga casi los mismos partidos de local que de visitante y nunca juegue más de dos seguidos en casa o fuera. Comprobar que cada pareja se enfrenta exactamente una vez y que nadie juega dos partidos en la misma jornada.

In [12]:
def calendario(equipos):
    equipos = ([None] if len(equipos) % 2 else []) + list(equipos)
    n = len(equipos)
    fijo, resto = equipos[0], equipos[1:]
    jornadas = []
    for r in range(n - 1):
        ronda = [fijo] + resto
        partidos = []
        for i in range(n // 2):
            local, visitante = ronda[i], ronda[n - 1 - i]
            if (r % 2 == 1) if i == 0 else (i % 2 == 1):
                local, visitante = visitante, local
            partidos.append((local, visitante))
        jornadas.append(partidos)
        resto = resto[-1:] + resto[:-1]
    return jornadas


def comprueba_calendario(equipos, jornadas):
    reales = [[p for p in jornada if None not in p] for jornada in jornadas]
    parejas = Counter(frozenset(p) for jornada in reales for p in jornada)
    assert set(parejas) == {frozenset(p) for p in itertools.combinations(equipos, 2)}
    assert set(parejas.values()) == {1}
    for jornada in reales:
        juegan = [e for p in jornada for e in p]
        assert len(juegan) == len(set(juegan))
    for equipo in equipos:
        en_casa = [p[0] == equipo for jornada in reales for p in jornada if equipo in p]
        assert abs(2 * sum(en_casa) - len(en_casa)) <= 1
        assert all(len(set(tramo)) > 1 for tramo in zip(en_casa, en_casa[1:], en_casa[2:]))


islas = ["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera", "El Hierro"]
jornadas = calendario(islas)
comprueba_calendario(islas, jornadas)
for numero, jornada in enumerate(jornadas, 1):
    partidos = [f"{l} - {v}" for l, v in jornada if None not in (l, v)]
    descansa = next((l or v for l, v in jornada if None in (l, v)), None)
    print(f"Jornada {numero}: " + ", ".join(partidos) + (f" (descansa {descansa})" if descansa else ""))

for n in range(2, 31):
    equipos = list(range(n))
    comprueba_calendario(equipos, calendario(equipos))

Jornada 1: La Gomera - Tenerife, Gran Canaria - La Palma, Fuerteventura - Lanzarote (descansa El Hierro)
Jornada 2: La Palma - El Hierro, Tenerife - Fuerteventura, Lanzarote - Gran Canaria (descansa La Gomera)
Jornada 3: Fuerteventura - La Gomera, El Hierro - Lanzarote, Gran Canaria - Tenerife (descansa La Palma)
Jornada 4: Lanzarote - La Palma, La Gomera - Gran Canaria, Tenerife - El Hierro (descansa Fuerteventura)
Jornada 5: Gran Canaria - Fuerteventura, La Palma - Tenerife, El Hierro - La Gomera (descansa Lanzarote)
Jornada 6: Tenerife - Lanzarote, Fuerteventura - El Hierro, La Gomera - La Palma (descansa Gran Canaria)
Jornada 7: El Hierro - Gran Canaria, Lanzarote - La Gomera, La Palma - Fuerteventura (descansa Tenerife)


## 13. Esquema genérico de divide y vencerás

Escribir `divide_y_venceras(problema, es_trivial, resolver_trivial, dividir, combinar)`, que resuelve directamente los problemas triviales y, si no, divide el problema, resuelve cada parte con el mismo esquema y combina las soluciones. Aplicarlo a dos problemas sobre tramos `(inicio, fin)` de una lista:

- el mínimo y el máximo a la vez, contando las comparaciones: con $n$ potencia de 2 son $\frac{3n}{2} - 2$, frente a las $2(n - 1)$ de buscarlos por separado;
- la suma máxima de un tramo contiguo, cuyo mejor tramo está en la mitad izquierda, en la derecha o cruza el centro (la mejor cola de la izquierda más la mejor cabeza de la derecha), en $O(n \log n)$. Comprobarlo contra la fuerza bruta.

In [13]:
def divide_y_venceras(problema, es_trivial, resolver_trivial, dividir, combinar):
    if es_trivial(problema):
        return resolver_trivial(problema)
    soluciones = [divide_y_venceras(parte, es_trivial, resolver_trivial, dividir, combinar)
                  for parte in dividir(problema)]
    return combinar(problema, soluciones)


def mitades(tramo):
    inicio, fin = tramo
    medio = (inicio + fin) // 2
    return [(inicio, medio), (medio, fin)]


def minimo_y_maximo(lista):
    comparaciones = 0

    def resolver_trivial(tramo):
        nonlocal comparaciones
        inicio, fin = tramo
        if fin - inicio == 1:
            return lista[inicio], lista[inicio]
        comparaciones += 1
        a, b = lista[inicio], lista[inicio + 1]
        return (a, b) if a <= b else (b, a)

    def combinar(tramo, soluciones):
        nonlocal comparaciones
        (min1, max1), (min2, max2) = soluciones
        comparaciones += 2
        return min(min1, min2), max(max1, max2)

    resultado = divide_y_venceras((0, len(lista)), lambda t: t[1] - t[0] <= 2, resolver_trivial, mitades, combinar)
    return resultado, comparaciones


rng = random.Random(3)
print(f"{'n':>6}{'divide y vencerás':>20}{'por separado':>15}")
for k in range(1, 11):
    n = 2 ** k
    lista = [rng.randint(-1000, 1000) for _ in range(n)]
    (minimo, maximo), comparaciones = minimo_y_maximo(lista)
    assert (minimo, maximo) == (min(lista), max(lista)) and comparaciones == 3 * n // 2 - 2
    print(f"{n:>6}{comparaciones:>20}{2 * (n - 1):>15}")
for n in range(1, 100):
    lista = [rng.randint(-50, 50) for _ in range(n)]
    assert minimo_y_maximo(lista)[0] == (min(lista), max(lista))

     n   divide y vencerás   por separado
     2                   1              2
     4                   4              6
     8                  10             14
    16                  22             30
    32                  46             62
    64                  94            126
   128                 190            254
   256                 382            510
   512                 766           1022
  1024                1534           2046


In [14]:
def suma_maxima(lista):
    def combinar(tramo, soluciones):
        inicio, fin = tramo
        medio = (inicio + fin) // 2
        cola = max(itertools.accumulate(reversed(lista[inicio:medio])))
        cabeza = max(itertools.accumulate(lista[medio:fin]))
        return max(*soluciones, cola + cabeza)

    return divide_y_venceras((0, len(lista)), lambda t: t[1] - t[0] == 1,
                             lambda t: lista[t[0]], mitades, combinar)


def suma_maxima_fuerza_bruta(lista):
    return max(sum(lista[i:j]) for i in range(len(lista)) for j in range(i + 1, len(lista) + 1))


caja = [120, -340, 210, 180, -90, 300, -510, 260, 40, -30, 150, -400]
print("Caja diaria:", caja)
print("Mejor racha de días seguidos:", suma_maxima(caja))
assert suma_maxima(caja) == suma_maxima_fuerza_bruta(caja)
for _ in range(500):
    lista = [rng.randint(-20, 20) for _ in range(rng.randint(1, 30))]
    assert suma_maxima(lista) == suma_maxima_fuerza_bruta(lista)

Caja diaria: [120, -340, 210, 180, -90, 300, -510, 260, 40, -30, 150, -400]
Mejor racha de días seguidos: 600


## 14. El rompecabezas del triominó

A un tablero de $2^n \times 2^n$ casillas le falta una. Cubrir el resto con triominós en L (tres casillas de un cuadrado de $2 \times 2$) sin que se solapen. Por divide y vencerás: se parte el tablero en cuatro cuadrantes; uno contiene la casilla que falta y en los otros tres se coloca un triominó en las tres casillas que rodean el centro, de modo que cada cuadrante queda como un tablero más pequeño al que le falta una casilla. Dibujar el resultado como texto, con una letra por ficha, y comprobar con todas las posiciones del hueco hasta $16 \times 16$ que cada ficha ocupa exactamente 3 casillas en L, que no queda ninguna sin cubrir y que hay $\frac{4^n - 1}{3}$ fichas.

In [15]:
def triomino(n, hueco):
    lado = 2 ** n
    tablero = [[0] * lado for _ in range(lado)]
    tablero[hueco[0]][hueco[1]] = -1
    numeros = itertools.count(1)

    def cubre(fila, columna, lado, hueco):
        if lado == 1:
            return
        mitad = lado // 2
        ficha = next(numeros)
        cuadrantes = []
        for df in (0, 1):
            for dc in (0, 1):
                f0, c0 = fila + df * mitad, columna + dc * mitad
                if f0 <= hueco[0] < f0 + mitad and c0 <= hueco[1] < c0 + mitad:
                    cuadrantes.append((f0, c0, hueco))
                else:
                    centro = (fila + mitad - 1 + df, columna + mitad - 1 + dc)
                    tablero[centro[0]][centro[1]] = ficha
                    cuadrantes.append((f0, c0, centro))
        for f0, c0, h in cuadrantes:
            cubre(f0, c0, mitad, h)

    cubre(0, 0, lado, hueco)
    return tablero


def dibuja_tablero(tablero):
    simbolos = string.ascii_uppercase + string.ascii_lowercase + string.digits
    return "\n".join(" ".join("#" if v < 0 else simbolos[(v - 1) % len(simbolos)] for v in fila) for fila in tablero)


def comprueba_triomino(tablero, hueco):
    casillas = defaultdict(list)
    for f, fila in enumerate(tablero):
        for c, v in enumerate(fila):
            casillas[v].append((f, c))
    assert casillas.pop(-1) == [hueco] and 0 not in casillas
    assert len(casillas) == (len(tablero) ** 2 - 1) // 3
    for celdas in casillas.values():
        filas, columnas = {f for f, _ in celdas}, {c for _, c in celdas}
        assert len(celdas) == 3 and len(filas) == len(columnas) == 2
        assert max(filas) - min(filas) == max(columnas) - min(columnas) == 1


tablero = triomino(3, (5, 2))
print(dibuja_tablero(tablero))
comprueba_triomino(tablero, (5, 2))
for n in range(1, 5):
    for hueco in itertools.product(range(2 ** n), repeat=2):
        comprueba_triomino(triomino(n, hueco), hueco)
comprueba_triomino(triomino(7, (100, 3)), (100, 3))

C C D D H H I I
C B B D H G G I
E B F F J J G K
E E F A A J K K
M M N N A R S S
M L # N R R Q S
O L L P T Q Q U
O O P P T T U U
